# Tiny: continuação até 2.000 atualizações
Execute com kernel **Python (protosam)** recém-iniciado. Encerre os kernels anteriores para liberar a GPU.
Este notebook restaura pesos, AdamW e RNG do piloto de 200 passos e executa mais 1.800 passos em uma nova pasta.
Mantém normalização RGB, último estágio 3, outnorm3 treinável, ALP e perdas do piloto. Não testa outras camadas ou normalizações.
Cada ciclo percorre as 1.304 imagens de treino em ordem sorteada; a região é sorteada novamente e pode coincidir com a anterior.
Validação fixa de pseudo-máscaras a cada 250 passos. Checkpoint de retomada a cada 50. Checkpoints 500/1000/1500/2000 preservados para avaliação posterior ALP + SAM. Não seleciona automaticamente o melhor modelo.


In [ ]:
from pathlib import Path
import sys
REPO = Path('/workspace/ProtoSAM-repro')
PACOTE = REPO / 'exp' / 'eft_tiny_2000_v1'
assert Path(sys.executable).resolve() == Path('/workspace/miniforge3/envs/protosam/bin/python').resolve(), sys.executable
assert (PACOTE / 'eft_continuar.py').is_file(), f'Extraia o pacote em {PACOTE}'
for caminho in (REPO, PACOTE):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))
from eft_continuar import executar
print('Python:', sys.executable)


Primeira execução: mantenha `RETOMAR = None`. Após interrupção, use a pasta **eft_tiny_2000_...** impressa abaixo e execute novamente. Não use a pasta do piloto.

In [ ]:
RETOMAR = None
# Exemplo de retomada: RETOMAR = '/workspace/ProtoSAM-repro/reproducao/eft_tiny_2000_...'
RESULTADOS = executar(retomar=RETOMAR)
print('RESULTADOS:', RESULTADOS)


In [ ]:
from pathlib import Path
import shutil
import ast

RAIZ = Path("/workspace/ProtoSAM-repro/exp")
V1 = RAIZ / "eft_tiny_2000_v1"
V2 = RAIZ / "eft_tiny_2000_v2"

assert (V1 / "eft_continuar.py").is_file(), f"Pacote não encontrado: {V1}"
V2.mkdir(exist_ok=True)

for nome in (
    "eft_dados.py",
    "eft_modelo.py",
    "eft_treino.py",
    "eft_plano_2000.py",
):
    shutil.copy2(V1 / nome, V2 / nome)

correcao = '''
from types import MethodType

class PrototiposVaziosNoAlinhamento(Exception):
    pass

def instalar(modelo):
    original_align = modelo.alignLoss
    original_prediction = modelo.cls_unit.get_prediction_from_prototypes
    estado = {"ativo": False, "ultimo": None}

    def prediction(self, prototypes, query, mode, *args, **kwargs):
        if estado["ativo"] and prototypes.shape[0] == 0:
            raise PrototiposVaziosNoAlinhamento(str(mode))
        return original_prediction(
            prototypes, query, mode, *args, **kwargs
        )

    def align(self, qry_fts, pred, supp_fts, fore_mask, back_mask):
        estado["ativo"] = True
        try:
            return original_align(
                qry_fts, pred, supp_fts, fore_mask, back_mask
            )
        except PrototiposVaziosNoAlinhamento as erro:
            estado["ultimo"] = {
                "motivo": "zero prototipos no alinhamento",
                "modo": str(erro),
                "acao": "alinhamento zero; CE preservada",
            }
            return qry_fts.sum() * 0.0
        finally:
            estado["ativo"] = False

    modelo.alignLoss = MethodType(align, modelo)
    modelo.cls_unit.get_prediction_from_prototypes = MethodType(
        prediction, modelo.cls_unit
    )
    return estado
'''

codigo = (V1 / "eft_continuar.py").read_text()

def substituir(antes, depois):
    global codigo
    assert codigo.count(antes) == 1, (
        "A versão local difere da esperada. Nenhum treinamento iniciado.\n"
        f"Trecho: {antes}"
    )
    codigo = codigo.replace(antes, depois, 1)

substituir(
    "from eft_plano_2000 import estender",
    "from eft_plano_2000 import estender\n"
    "from eft_alinhamento_seguro import instalar",
)

substituir(
    "def executar(retomar=None, piloto=",
    "def executar(retomar=None, continuar_de=None, piloto=",
)

substituir(
    "    fontes = sorted",
    '''
    if retomar:
        anterior = json.loads((Path(retomar) / "config.json").read_text())
        continuar_de = anterior["continuacao_origem"]

    if continuar_de is None:
        candidatas = []
        for f in sorted(
            (repo / "reproducao").glob("eft_tiny_2000_*/config.json")
        ):
            c = json.loads(f.read_text())
            if (
                c.get("versao") == 2
                and c.get("variante") == "tiny"
                and (f.parent / "checkpoint_latest.pt").is_file()
            ):
                candidatas.append(f.parent)

        assert len(candidatas) == 1, (
            "Informe CONTINUAR_DE com a pasta da execução que falhou. "
            f"Candidatas: {candidatas}"
        )
        continuar_de = candidatas[0]

    continuar_de = Path(continuar_de).resolve()
    config_anterior = json.loads(
        (continuar_de / "config.json").read_text()
    )
    assert config_anterior["versao"] == 2
    assert config_anterior["variante"] == "tiny"
    assert config_anterior["plano_estendido_sha256"] == assinatura(planos)
    assert config_anterior["piloto_origem"] == str(piloto)
    assert config_anterior["ambiente"] == ambiente

    for k in (
        "lr", "betas", "eps", "weight_decay", "scheduler", "amp",
        "estagio", "normalizacao", "resolucao", "grade_alp",
        "kernel_alp", "loss",
    ):
        assert config_anterior[k] == origem[k], k

    for arquivo, digest in config_anterior["fontes_continuacao_sha256"].items():
        assert sha256(arquivo) == digest, f"Fonte v1 alterada: {arquivo}"

    fontes = sorted''',
)

substituir(
    "config.update(versao=2,",
    '''config.update(
                  versao=3,
                  continuacao_origem=str(continuar_de),
                  checkpoint_continuacao_sha256=sha256(
                      continuar_de / "checkpoint_latest.pt"
                  ),
                  alinhamento_degenerado=(
                      "Se zero prototipos no alinhamento: "
                      "align=0, CE ativa, registrar episodio"
                  ),''',
)

substituir(
    "f'eft_tiny_2000_{agora()}'",
    "f'eft_tiny_2000_v2_{agora()}'",
)

substituir(
    "        proprio = pasta / 'checkpoint_latest.pt'",
    "        estado_align = instalar(modelo)\n"
    "        proprio = pasta / 'checkpoint_latest.pt'",
)

substituir(
    "proprio if proprio.exists() else piloto / 'checkpoint_latest.pt'",
    "proprio if proprio.exists() else continuar_de / 'checkpoint_latest.pt'",
)

substituir(
    "(config if proprio.exists() else origem)",
    "(config if proprio.exists() else config_anterior)",
)

substituir(
    "        if not proprio.exists():\n            assert passo == 200",
    '''
        if not proprio.exists():
            for n in [200] + list(range(250, passo + 1, 250)):
                f = continuar_de / f"validacao_{n:04d}.json"
                if f.exists():
                    shutil.copy2(f, pasta / f.name)

            gravar_json(
                pasta / "transicao.json",
                {
                    "passo_origem": passo,
                    "primeiro_passo_com_nova_regra": passo + 1,
                    "origem": str(continuar_de),
                },
            )''',
)

substituir(
    "            retorno = modelo([[s]]",
    '            estado_align["ultimo"] = None\n'
    "            retorno = modelo([[s]]",
)

substituir(
    "'regiao': r['regiao'], 'ce':",
    "'regiao': r['regiao'], "
    "'alinhamento_excepcao': estado_align['ultimo'], 'ce':",
)

substituir(
    "            passo = numero\n",
    '''
            if estado_align["ultimo"] is not None:
                print(
                    f"Passo {numero}: alinhamento sem protótipos; "
                    "atualização com CE. Ocorrência registrada.",
                    flush=True,
                )
            passo = numero
''',
)

substituir(
    "'limite':'Validação",
    "'regra_degenerada': config['alinhamento_degenerado'],\n"
    "                  'checkpoints_anteriores_em': str(continuar_de),\n"
    "                  'limite':'Validação",
)

# Verifica a sintaxe antes de salvar os módulos.
ast.parse(correcao)
ast.parse(codigo)

(V2 / "eft_alinhamento_seguro.py").write_text(correcao)
(V2 / "eft_continuar.py").write_text(codigo)

print("Correção preparada em:", V2)
print("Nenhum treinamento iniciado. Execute a próxima célula.")

In [ ]:
from pathlib import Path
import sys
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PACOTE = REPO / "exp" / "eft_tiny_2000_v2"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), f"Selecione o kernel Python (protosam). Atual: {sys.executable}"

for caminho in (REPO, PACOTE):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

# Garante que os módulos sejam importados da pasta corrigida.
for nome in (
    "eft_continuar",
    "eft_alinhamento_seguro",
    "eft_plano_2000",
    "eft_treino",
    "eft_modelo",
    "eft_dados",
):
    sys.modules.pop(nome, None)

importlib.invalidate_caches()
from eft_continuar import executar

# Localiza automaticamente a execução v1 se houver uma única candidata.
CONTINUAR_DE = None

# Primeira execução da correção: None.
# Se esta v2 for interrompida, coloque aqui o caminho que ela imprimir.
RETOMAR = None

RESULTADOS = executar(
    continuar_de=CONTINUAR_DE,
    retomar=RETOMAR,
)
print("RESULTADOS:", RESULTADOS)